# Dayanıklı (Resilient) Bir API İstemcisi İnşa Etmek

**İş Problemi:** [22.5.1-Python_ile_API_Tüketmek.pdf](22.5.1-Python_ile_API_Tüketmek.pdf) dosyasında
anlattığımız, production kalitesindeki bir API istemcisinin sahip olması gereken 5 özelliği
(session kullanımı, akıllı tekrar deneme, hata ayrımı, önbellekleme, otomatik sayfalama) uçtan uca,
çalışan bir Python sınıfına dökmek.

Bu notebook'u TAMAMEN OFFLINE ve HERKESTE AYNI ŞEKİLDE ÇALIŞACAK hale getirmek için, dış bir genel
API'ye bağlanmak yerine kendi basit "Kitap Kataloğu" API'mizi yerel bir portta (127.0.0.1) arka plan
iş parçacığında (thread) ayağa kaldırıyoruz ve istemcimizi buna karşı test ediyoruz.

Gerekli kütüphaneyi kurmak için: `pip install requests`

## 1. Sahte (Mock) Bir "Kitap Kataloğu" API'si Kurmak

In [1]:
import json
import random
import threading
import time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import urlparse, parse_qs

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# 47 kitaplık sahte bir katalog - gerçekçi bir sayfalama senaryosu için.
TUM_KITAPLAR = [
    {"id": i, "baslik": f"Kitap #{i}", "yazar": f"Yazar {chr(65 + i % 26)}"}
    for i in range(1, 48)
]
SAYFA_BOYUTU = 10

kararsiz_endpoint_sayaci = {"deneme": 0}
hiz_siniri_sayaci = {"istek": 0}

C:\Users\ONI\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [2]:
class KitapKatalogHandler(BaseHTTPRequestHandler):
    def log_message(self, format, *args):
        pass  # Konsolu kirletmemek için sunucunun kendi loglarını susturuyoruz.

    def _json_yanit(self, status_code, payload, extra_headers=None):
        body = json.dumps(payload, ensure_ascii=False).encode("utf-8")
        self.send_response(status_code)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        for k, v in (extra_headers or {}).items():
            self.send_header(k, v)
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        parsed = urlparse(self.path)
        qs = parse_qs(parsed.query)

        if parsed.path == "/kitaplar":
            sayfa = int(qs.get("sayfa", ["1"])[0])
            baslangic = (sayfa - 1) * SAYFA_BOYUTU
            bitis = baslangic + SAYFA_BOYUTU
            sayfa_verisi = TUM_KITAPLAR[baslangic:bitis]
            self._json_yanit(200, {
                "kitaplar": sayfa_verisi,
                "sayfa": sayfa,
                "sonraki_sayfa_var_mi": bitis < len(TUM_KITAPLAR),
            })

        elif parsed.path.startswith("/kitaplar/"):
            kitap_id = int(parsed.path.split("/")[-1])
            eslesen = next((k for k in TUM_KITAPLAR if k["id"] == kitap_id), None)
            if eslesen:
                self._json_yanit(200, eslesen)
            else:
                self._json_yanit(404, {"hata": "Kitap bulunamadı", "kitap_id": kitap_id})

        elif parsed.path == "/kararsiz":
            # İlk 2 çağrıda bilerek 503 döner, 3. çağrıda başarılı olur (geçici sunucu arızası simülasyonu).
            kararsiz_endpoint_sayaci["deneme"] += 1
            if kararsiz_endpoint_sayaci["deneme"] < 3:
                self._json_yanit(503, {"hata": "Sunucu geçici olarak meşgul, tekrar deneyin"})
            else:
                self._json_yanit(200, {"mesaj": "Başarılı!", "kacinci_denemede": kararsiz_endpoint_sayaci["deneme"]})

        elif parsed.path == "/hiz-siniri":
            # Her 3 istekten birinde 429 döner - hız sınırlama simülasyonu.
            hiz_siniri_sayaci["istek"] += 1
            if hiz_siniri_sayaci["istek"] % 3 == 0:
                self._json_yanit(429, {"hata": "Çok fazla istek"}, extra_headers={"Retry-After": "1"})
            else:
                self._json_yanit(200, {"mesaj": "OK", "istek_no": hiz_siniri_sayaci["istek"]})

        else:
            self._json_yanit(404, {"hata": "Endpoint bulunamadı"})


def sunucuyu_baslat():
    server = ThreadingHTTPServer(("127.0.0.1", 0), KitapKatalogHandler)
    port = server.server_address[1]
    thread = threading.Thread(target=server.serve_forever, daemon=True)
    thread.start()
    return server, port


sunucu, PORT = sunucuyu_baslat()
BASE_URL = f"http://127.0.0.1:{PORT}"
print(f"Sahte Kitap Kataloğu API'si başlatıldı: {BASE_URL}")

Sahte Kitap Kataloğu API'si başlatıldı: http://127.0.0.1:59637


## 2. Dayanıklı API İstemcisi Sınıfı

[22.5.1, Bölüm 3](22.5.1-Python_ile_API_Tüketmek.pdf)'te anlatılan 5 prensibi uygular: (1) Session ile
bağlantı yeniden kullanımı, (2) otomatik + akıllı tekrar deneme (sadece 429/5xx'te, exponential backoff
ile), (3) 4xx hatalarında tekrar denememe, (4) basit TTL önbellek, (5) otomatik sayfalama (generator).

In [3]:
class ResilientAPIClient:
    def __init__(self, base_url, max_deneme=4, cache_ttl_saniye=30, zaman_asimi=5):
        self.base_url = base_url.rstrip("/")
        self.zaman_asimi = zaman_asimi
        self.cache_ttl = cache_ttl_saniye
        self._cache = {}  # {url: (deger, kaydedilme_zamani)}

        self.session = requests.Session()

        # urllib3'ün Retry mekanizması: SADECE 429 ve 5xx durum kodlarında, üstel geri çekilmeyle
        # otomatik tekrar dener. 4xx (400, 401, 404 gibi) kalıcı hatalarda HİÇ tekrar denemez.
        retry_stratejisi = Retry(
            total=max_deneme,
            backoff_factor=0.3,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=["GET"],
            respect_retry_after_header=True,
            raise_on_status=False,
        )
        adapter = HTTPAdapter(max_retries=retry_stratejisi)
        self.session.mount("http://", adapter)
        self.session.mount("https://", adapter)

    def _cache_oku(self, url):
        if url in self._cache:
            deger, kayit_zamani = self._cache[url]
            if time.time() - kayit_zamani < self.cache_ttl:
                return deger
            del self._cache[url]
        return None

    def get(self, path, params=None, cache_kullan=True):
        url = f"{self.base_url}{path}"
        cache_anahtari = f"{url}?{params}"

        if cache_kullan:
            onbellek_sonucu = self._cache_oku(cache_anahtari)
            if onbellek_sonucu is not None:
                return onbellek_sonucu

        yanit = self.session.get(url, params=params, timeout=self.zaman_asimi)

        if cache_kullan and yanit.status_code == 200:
            self._cache[cache_anahtari] = (yanit, time.time())

        return yanit

    def tum_kitaplari_getir(self):
        """Sayfalama detaylarını çağıran koddan tamamen gizleyen bir generator."""
        sayfa = 1
        while True:
            yanit = self.get("/kitaplar", params={"sayfa": sayfa}, cache_kullan=False)
            yanit.raise_for_status()
            veri = yanit.json()
            for kitap in veri["kitaplar"]:
                yield kitap
            if not veri["sonraki_sayfa_var_mi"]:
                break
            sayfa += 1


client = ResilientAPIClient(BASE_URL)
print("İstemci hazır.")

İstemci hazır.


## 3. Manuel Retry Döngüsü (Mekanizmayı Şeffaf Göstermek İçin)

`HTTPAdapter` + `Retry` "sihirli" görünebilir; ne yaptığını somut görmek için AYNI mantığı elle de
yazıyoruz (bkz. 22.5.1, Bölüm 3.2 - üstel geri çekilme formülü).

In [4]:
def manuel_retry_ile_istek(url, max_deneme=4, taban_bekleme=0.3):
    for deneme in range(1, max_deneme + 1):
        yanit = requests.get(url, timeout=5)

        if yanit.status_code < 400:
            return yanit

        if yanit.status_code < 500 and yanit.status_code != 429:
            print(f"  [Deneme {deneme}] Kalıcı hata ({yanit.status_code}) - tekrar denenmiyor.")
            return yanit

        bekleme_suresi = taban_bekleme * (2 ** (deneme - 1))
        if "Retry-After" in yanit.headers:
            bekleme_suresi = float(yanit.headers["Retry-After"])
        print(f"  [Deneme {deneme}] Geçici hata ({yanit.status_code}) - {bekleme_suresi:.1f}sn bekleyip tekrar denenecek.")
        time.sleep(bekleme_suresi)

    return yanit

## 4. Testler\n\n### Test 1 — Basit GET İsteği

In [5]:
yanit = client.get("/kitaplar/5")
assert yanit.status_code == 200
kitap = yanit.json()
print(f"5 numaralı kitap: {kitap}")
assert kitap["id"] == 5

5 numaralı kitap: {'id': 5, 'baslik': 'Kitap #5', 'yazar': 'Yazar F'}


### Test 2 — 404 (Kalıcı Hata): Tekrar Deneme YAPILMAMALI

In [6]:
baslangic = time.time()
yanit = client.get("/kitaplar/9999")
sure = time.time() - baslangic
assert yanit.status_code == 404
print(f"404 yanıtı {sure:.3f} saniyede geldi (tekrar deneme YOK, çok hızlı olmalı) -> {yanit.json()}")
assert sure < 1.0, "404'te tekrar deneme yapılmamalı, çok uzun sürdü!"

404 yanıtı 0.003 saniyede geldi (tekrar deneme YOK, çok hızlı olmalı) -> {'hata': 'Kitap bulunamadı', 'kitap_id': 9999}


### Test 3 — Kararsız Endpoint: `HTTPAdapter` Otomatik Tekrar Deniyor

In [7]:
kararsiz_endpoint_sayaci["deneme"] = 0
baslangic = time.time()
yanit = client.get("/kararsiz", cache_kullan=False)
sure = time.time() - baslangic
assert yanit.status_code == 200, "Otomatik retry başarısız oldu!"
print(f"Kararsız endpoint {sure:.2f} saniye içinde, otomatik tekrar denemelerle başarılı oldu: {yanit.json()}")

Kararsız endpoint 0.61 saniye içinde, otomatik tekrar denemelerle başarılı oldu: {'mesaj': 'Başarılı!', 'kacinci_denemede': 3}


### Test 4 — Manuel Retry Döngüsü ile Aynı Senaryo

In [8]:
kararsiz_endpoint_sayaci["deneme"] = 0
yanit = manuel_retry_ile_istek(f"{BASE_URL}/kararsiz")
assert yanit.status_code == 200
print(f"Manuel retry sonucu: {yanit.json()}")

  [Deneme 1] Geçici hata (503) - 0.3sn bekleyip tekrar denenecek.


  [Deneme 2] Geçici hata (503) - 0.6sn bekleyip tekrar denenecek.


Manuel retry sonucu: {'mesaj': 'Başarılı!', 'kacinci_denemede': 3}


### Test 5 — Önbellekleme (Cache): İkinci Çağrı Ağa Gitmemeli

In [9]:
client.get("/kitaplar/1")  # ilk çağrı - önbelleğe yazılır
baslangic = time.time()
yanit_cache = client.get("/kitaplar/1")  # ikinci çağrı - önbellekten dönmeli
sure_cache = time.time() - baslangic
print(f"Önbellekten okuma süresi: {sure_cache*1000:.2f} ms (gerçek bir ağ isteğinden çok daha hızlı olmalı)")
assert yanit_cache.json()["id"] == 1

Önbellekten okuma süresi: 0.31 ms (gerçek bir ağ isteğinden çok daha hızlı olmalı)


### Test 6 — Otomatik Sayfalama (Generator): TÜM 47 Kitabı Tek Döngüyle Çekmek

In [10]:
tum_kitaplar_listesi = list(client.tum_kitaplari_getir())
print(f"Toplam çekilen kitap sayısı: {len(tum_kitaplar_listesi)}")
assert len(tum_kitaplar_listesi) == 47, "Tüm kitaplar çekilemedi!"
print(f"İlk kitap: {tum_kitaplar_listesi[0]['baslik']}, Son kitap: {tum_kitaplar_listesi[-1]['baslik']}")

Toplam çekilen kitap sayısı: 47
İlk kitap: Kitap #1, Son kitap: Kitap #47


### Test 7 — Hız Sınırı (429): `Retry-After`'a Saygı Gösterilerek Otomatik Bekleniyor

In [11]:
hiz_siniri_sayaci["istek"] = 0
basarili_istek_sayisi = 0
baslangic = time.time()
for _ in range(5):
    yanit = client.get("/hiz-siniri", cache_kullan=False)
    if yanit.status_code == 200:
        basarili_istek_sayisi += 1
sure = time.time() - baslangic
print(f"5 isteğin {basarili_istek_sayisi} tanesi başarılı oldu (429'lar otomatik retry ile aşıldı), toplam süre: {sure:.2f}sn")
assert basarili_istek_sayisi == 5, "Hız sınırı aşılamadı!"

sunucu.shutdown()
print("\nTÜM TESTLER BAŞARIYLA TAMAMLANDI.")

5 isteğin 5 tanesi başarılı oldu (429'lar otomatik retry ile aşıldı), toplam süre: 2.02sn



TÜM TESTLER BAŞARIYLA TAMAMLANDI.


## Sonuç

Production kalitesinde bir API istemcisinin 5 temel özelliğini (session, akıllı tekrar deneme, hata
ayrımı, önbellekleme, otomatik sayfalama) sıfırdan inşa ettik ve her birini GERÇEK (ama tamamen yerel
ve tekrar üretilebilir) hata senaryolarıyla test ettik.

Sıradaki dosya: [22.6.1-Python_ile_API_Geliştirmek_FastAPI.pdf](../22.6-Python_ile_API_Geliştirmek/22.6.1-Python_ile_API_Geliştirmek_FastAPI.pdf)
ve [22.6.2_kutuphane_api.ipynb](../22.6-Python_ile_API_Geliştirmek/22.6.2_kutuphane_api.ipynb) — bu
sefer bir API'yi TÜKETMEK değil, sıfırdan İNŞA ETMEK.